# 🚢 Titanic Survival Predictor
### Machine Learning + Data Analysis Project
**Author:** Vaibhav Lohchab 
**Tools:** Python, Pandas, Scikit-learn, Seaborn, Matplotlib

---
This project analyzes the Titanic dataset to:
- Perform complete Exploratory Data Analysis (EDA)
- Clean and engineer features
- Train and compare multiple ML models
- Evaluate and visualize results


## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install any missing libraries
!pip install seaborn scikit-learn pandas matplotlib numpy --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

# Style settings
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('✅ All libraries imported successfully!')

## 📂 Step 2: Load the Dataset

In [ ]:
# Load Titanic dataset directly from URL (no download needed)
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

print(f'✅ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(10)

## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('--- Dataset Info ---')
print(df.info())
print('\n--- Missing Values ---')
print(df.isnull().sum())
print('\n--- Basic Statistics ---')
df.describe()

In [ ]:
# 1. Overall Survival Rate
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

survival_counts = df['Survived'].value_counts()
axes[0].pie(
    survival_counts,
    labels=['Did Not Survive', 'Survived'],
    autopct='%1.1f%%',
    colors=['#e74c3c', '#2ecc71'],
    startangle=90,
    explode=(0.05, 0.05)
)
axes[0].set_title('Overall Survival Rate', fontsize=14, fontweight='bold')

# 2. Survival by Gender
sns.barplot(data=df, x='Sex', y='Survived', palette=['#3498db', '#e91e63'], ax=axes[1])
axes[1].set_title('Survival Rate by Gender', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Survival Rate')
axes[1].set_ylim(0, 1)

plt.suptitle('Titanic Survival Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('survival_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Chart saved!')

In [ ]:
# 3. Survival by Passenger Class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df, x='Pclass', y='Survived', palette='Blues_d', ax=axes[0])
axes[0].set_title('Survival by Passenger Class', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Passenger Class (1=1st, 2=2nd, 3=3rd)')
axes[0].set_ylabel('Survival Rate')

# 4. Age Distribution
df['Age'].dropna().hist(bins=30, color='#9b59b6', edgecolor='white', ax=axes[1])
axes[1].set_title('Age Distribution of Passengers', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('class_age_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5. Heatmap: Survival by Class & Gender
pivot = df.pivot_table(values='Survived', index='Sex', columns='Pclass', aggfunc='mean')

plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', linewidths=0.5,
            cbar_kws={'label': 'Survival Rate'})
plt.title('Survival Rate by Gender & Passenger Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('heatmap_survival.png', dpi=150, bbox_inches='tight')
plt.show()

## 🛠️ Step 4: Data Cleaning & Feature Engineering

In [ ]:
# Work on a copy
data = df.copy()

# --- Fill Missing Values ---
data['Age'].fillna(data['Age'].median(), inplace=True)
data['Embarked'].fillna(data['Embarked'].mode()[0], inplace=True)
data.drop(columns=['Cabin'], inplace=True)  # Too many missing values

# --- Feature Engineering ---
# Family size
data['FamilySize'] = data['SibSp'] + data['Parch'] + 1

# Is Alone?
data['IsAlone'] = (data['FamilySize'] == 1).astype(int)

# Title from name
data['Title'] = data['Name'].str.extract(r' ([A-Za-z]+)\.')
data['Title'] = data['Title'].replace(
    ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare'
)
data['Title'] = data['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# Age group
data['AgeGroup'] = pd.cut(
    data['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior']
)

# Fare group
data['FareBand'] = pd.qcut(data['Fare'], 4, labels=['Low', 'Mid', 'High', 'VeryHigh'])

print('✅ Feature engineering complete!')
print(f'New columns: {["FamilySize", "IsAlone", "Title", "AgeGroup", "FareBand"]}')
data[['Name','Title','Age','AgeGroup','FamilySize','IsAlone','FareBand']].head()

In [ ]:
# --- Encode Categorical Features ---
le = LabelEncoder()

for col in ['Sex', 'Embarked', 'Title', 'AgeGroup', 'FareBand']:
    data[col] = le.fit_transform(data[col].astype(str))

# Select final features
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
            'Embarked', 'FamilySize', 'IsAlone', 'Title', 'AgeGroup', 'FareBand']

X = data[features]
y = data['Survived']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'✅ Data split → Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 🤖 Step 5: Train & Compare Multiple ML Models

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}

print('Training models...\n')
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cv  = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy').mean()
    results[name] = {'Test Accuracy': acc, 'CV Accuracy': cv, 'model': model, 'preds': y_pred}
    print(f'  {name:<25} | Test: {acc:.4f} | CV: {cv:.4f}')

print('\n✅ All models trained!')

In [ ]:
# Model Comparison Chart
model_names = list(results.keys())
test_accs   = [results[m]['Test Accuracy'] for m in model_names]
cv_accs     = [results[m]['CV Accuracy']   for m in model_names]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, test_accs, width, label='Test Accuracy',  color='#3498db', alpha=0.85)
bars2 = ax.bar(x + width/2, cv_accs,   width, label='CV Accuracy (5-fold)', color='#e67e22', alpha=0.85)

ax.set_xlabel('Model')
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison: Test vs Cross-Validation Accuracy', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15)
ax.set_ylim(0.7, 0.9)
ax.legend()
ax.bar_label(bars1, fmt='%.3f', padding=3)
ax.bar_label(bars2, fmt='%.3f', padding=3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 📊 Step 6: Evaluate the Best Model

In [ ]:
# Find best model
best_name = max(results, key=lambda k: results[k]['CV Accuracy'])
best      = results[best_name]

print(f'🏆 Best Model: {best_name}')
print(f'   Test Accuracy : {best["Test Accuracy"]:.4f}')
print(f'   CV Accuracy   : {best["CV Accuracy"]:.4f}')
print()
print('--- Classification Report ---')
print(classification_report(y_test, best['preds'], target_names=['Not Survived', 'Survived']))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, best['preds'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Survived', 'Survived'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance (Random Forest)
rf_model = results['Random Forest']['model']
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(9, 6))
importances.plot(kind='barh', color='#1abc9c', edgecolor='white')
plt.title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔮 Step 7: Make a Custom Prediction

In [ ]:
# Predict survival for a custom passenger
# Try changing these values!
passenger = {
    'Pclass':     3,       # 1=First, 2=Second, 3=Third
    'Sex':        0,       # 0=Female, 1=Male
    'Age':        22,
    'SibSp':      1,       # Siblings / Spouses aboard
    'Parch':      0,       # Parents / Children aboard
    'Fare':       7.25,
    'Embarked':   2,       # 0=C, 1=Q, 2=S
    'FamilySize': 2,
    'IsAlone':    0,
    'Title':      2,       # 0=Master,1=Miss,2=Mr,3=Mrs,4=Rare
    'AgeGroup':   2,       # 0=Adult,1=Child,2=Senior,3=Teen,4=YoungAdult
    'FareBand':   0        # 0=High,1=Low,2=Mid,3=VeryHigh
}

passenger_df = pd.DataFrame([passenger])
passenger_scaled = scaler.transform(passenger_df)

prediction = best['model'].predict(passenger_scaled)[0]
probability = best['model'].predict_proba(passenger_scaled)[0]

print(f'🚢 Prediction using: {best_name}')
print(f'   Survival Prediction : {"✅ SURVIVED" if prediction == 1 else "❌ DID NOT SURVIVE"}')
print(f'   Probability of Survival  : {probability[1]*100:.1f}%')
print(f'   Probability of Not Surviving: {probability[0]*100:.1f}%')

## ✅ Step 8: Summary

| Step | What We Did |
|------|-------------|
| EDA  | Explored survival by gender, class, age |
| Cleaning | Filled missing values, dropped high-null columns |
| Feature Engineering | Created FamilySize, Title, AgeGroup, FareBand |
| Modeling | Trained 4 models: LR, DT, RF, GB |
| Evaluation | Compared accuracy, CV score, confusion matrix |
| Prediction | Built a custom passenger predictor |

### Key Insights
- **Women had a much higher survival rate** than men
- **1st class passengers** survived more than 2nd or 3rd class
- **Children** had better survival chances
- **Title** and **Fare** are among the most important features
- **Random Forest / Gradient Boosting** performed best overall
